In [ ]:
#импорты по красоте
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

import numpy as np
import re
from sklearn.model_selection import GroupShuffleSplit
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_extraction.text import TfidfVectorizer
import math

import time

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score



In [ ]:
BASE = Path("dataset")

df = pd.read_csv(BASE / "dataset.csv")
df

In [ ]:
print("Размер:", df.shape)
print()
print(df["cwe"].value_counts())
print()
print(df["label"].value_counts())

In [ ]:
df["exists"] = df["path"].apply(lambda p: Path(p).exists())
df["exists"].value_counts()

In [ ]:
sample_path = Path(df.iloc[0]["path"])
print(sample_path)

## Собственно вот оно и представление !!!

In [ ]:
text = sample_path.read_text()
print(text[:2000])

## инструкции LLVM, которые считаю

In [ ]:
OPCODES = [
    "alloca",
    "load",
    "store",
    "getelementptr",
    "call",
    "invoke",
    "br",
    "switch",
    "phi",
    "select",
    "icmp",
    "fcmp",
    "add",
    "sub",
    "mul",
    "udiv",
    "sdiv",
    "fdiv",
    "and",
    "or",
    "xor",
    "shl",
    "lshr",
    "ashr",
    "trunc",
    "zext",
    "sext",
    "ptrtoint",
    "inttoptr",
    "bitcast",
    "ret",
]

## функция, которая превращает один .ll в набор чисел:

In [ ]:
def extract_llvm_stats(path):
    text = Path(path).read_text(errors="ignore") # дешевле просто игнорить пока случаи с траблами
    lines = text.splitlines()

    features = {} # словарь фичей

    # количество фунций с телом

    features["functions"] = sum(
        1 for line in lines
        if line.lstrip().startswith("define ")
    )

    # Очень простая оценка числа basic blocks:
    # метки вроде entry:, if.then:, if.end:
    features["basic_blocks"] = sum(
        1 for line in lines
        if re.match(r"^\s*[A-Za-z$._][A-Za-z0-9$._-]*:\s*(;.*)?$", line)
    )

    # Счётчики LLVM-инструкций
    for opcode in OPCODES:
        features[opcode] = 0

    total_instructions = 0

    for line in lines:
        line = line.strip()

        if not line:
            continue

        if line.startswith(";"):
            continue

        # Отбрасываем декларации, определения функций и служебные строки
        if line.startswith((
            "define ",
            "declare ",
            "source_filename",
            "target ",
            "attributes ",
            "!",
            "@"
        )):
            continue

        # У инструкции может быть результат:
        # %5 = load ...
        #
        # либо результата может не быть:
        # store ...
        # br ...
        # ret ...

        if "=" in line:
            instruction = line.split("=", 1)[1].strip()
        else:
            instruction = line

        if not instruction:
            continue

        opcode = instruction.split()[0]

        if opcode in OPCODES:
            features[opcode] += 1
            total_instructions += 1

    features["instructions"] = total_instructions

    return features

In [ ]:
stats = extract_llvm_stats(sample_path)
stats

## Датасет полный с фичами

In [ ]:
feature_rows = []

for i, row in df.iterrows():
    features = extract_llvm_stats(row["path"])
    feature_rows.append(features)

features_df = pd.DataFrame(feature_rows)

features_df.head()

In [ ]:
features_df.shape

In [ ]:
data_stats = pd.concat(
    [
        df[["cwe", "case", "label", "target"]].reset_index(drop=True),
        features_df.reset_index(drop=True),
    ],
    axis=1,
)

data_stats

# разбиваем на X и y


In [ ]:
FEAUTURE_COLUMNS = list(features_df.columns)

X = data_stats[FEAUTURE_COLUMNS]
y = data_stats["target"]

print(X.shape)
print(y.shape)

## Используем GroupShuffleSplit т.к. на классы можно разбить по разным меткам

In [ ]:
groups = data_stats["cwe"] + "::" + data_stats["case"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=.2,
    random_state=42 #надеюсь, что бдует воспроизводимо
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train", X_train.shape)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train)

fig, ax = plt.subplots(figsize=(8, 6))

good = y_train == 0
bad = y_train == 1

ax.scatter(
    X_pca[good, 0],
    X_pca[good, 1],
    s=10,
    alpha=0.7,
    label="good"
)

ax.scatter(
    X_pca[bad, 0],
    X_pca[bad, 1],
    s=10,
    alpha=0.7,
    label="bad"
)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("PCA для LLVM stats")
ax.legend()

plt.show()

## Логистическая регрессия

In [ ]:


model = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("clf", LogisticRegression(
        C=1.0,
        max_iter=5000,
        random_state=42,
    )),
])

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

In [ ]:

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1       :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_score))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["good", "bad"]
).plot()

In [ ]:
tn, fp, fn, tp = cm.ravel()

print("True Negative :", tn)
print("False Positive:", fp)
print("False Negative:", fn)
print("True Positive :", tp)

## Теперь опкоды тф-идф

In [ ]:
OPCODE_SET = set(OPCODES)

def extract_opcode_sequence(path):
    text = Path(path).read_text(errors="ignore")

    sequence = []

    for raw_line in text.splitlines():

        line = raw_line.strip()

        if not line:
            continue

        if line.startswith(";"):
            continue

        if line.startswith((
            "define ",
            "declare ",
            "source_filename",
            "target ",
            "attributes ",
            "!",
            "@"
        )):
            continue

        # Если инструкция сохраняет результат:
        # %8 = load ...
        # %9 = call ...
        if "=" in line:
            instruction = line.split("=", 1)[1].strip()
        else:
            instruction = line

        tokens = instruction.split()

        if not tokens:
            continue

        opcode = tokens[0]

        # На случай LLVM-вариантов:
        # tail call ...
        # musttail call ...
        if opcode in {"tail", "musttail", "notail"}:
            if len(tokens) > 1 and tokens[1] == "call":
                opcode = "call"

        if opcode in OPCODE_SET:
            sequence.append(opcode)

    return sequence

In [ ]:
seq = extract_opcode_sequence(df.iloc[0]["path"])

print(seq[:50])
print()
print("Количество инструкций:", len(seq))

## текст для каждого из 4704 файлов:

In [ ]:
opcode_texts = []

for path in df["path"]:
    seq = extract_opcode_sequence(path)
    opcode_texts.append(" ".join(seq))

print(opcode_texts[0][:500])

In [ ]:
train_texts = [opcode_texts[i] for i in train_idx]
test_texts = [opcode_texts[i] for i in test_idx]

y_train_tfidf = y.iloc[train_idx]
y_test_tfidf = y.iloc[test_idx]

 ngram_range=(1, 2) означает, что модель получает и отдельные инструкции:
- load
- call
- br

и пары соседних инструкций:

- load call
- call store
- icmp br

In [ ]:


tfidf = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=2,
)

In [ ]:
X_train_tfidf = tfidf.fit_transform(train_texts)
X_test_tfidf = tfidf.transform(test_texts)

print("Train:", X_train_tfidf.shape)
print("Test :", X_test_tfidf.shape)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train_tfidf)

fig, ax = plt.subplots(figsize=(8, 6))

good = y_train == 0
bad = y_train == 1

ax.scatter(
    X_pca[good, 0],
    X_pca[good, 1],
    s=10,
    alpha=0.7,
    label="good"
)

ax.scatter(
    X_pca[bad, 0],
    X_pca[bad, 1],
    s=10,
    alpha=0.7,
    label="bad"
)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("PCA для Opcode TF-IDF")
ax.legend()

plt.show()

# Вот здесь уже буквально видно, что представляют признаки модели.

In [ ]:
feature_names = tfidf.get_feature_names_out()

print("Количество TF-IDF признаков:", len(feature_names))
print(feature_names[:50])

In [ ]:
from sklearn.linear_model import LogisticRegression

tfidf_model = LogisticRegression(
    C=1.0,
    max_iter=5000,
    random_state=42,
)

tfidf_model.fit(
    X_train_tfidf,
    y_train_tfidf
)

In [ ]:
y_pred_tfidf = tfidf_model.predict(X_test_tfidf)

y_score_tfidf = tfidf_model.predict_proba(
    X_test_tfidf
)[:, 1]

In [ ]:
print("Accuracy :", accuracy_score(y_test_tfidf, y_pred_tfidf))
print("Precision:", precision_score(y_test_tfidf, y_pred_tfidf))
print("Recall   :", recall_score(y_test_tfidf, y_pred_tfidf))
print("F1       :", f1_score(y_test_tfidf, y_pred_tfidf))
print("ROC-AUC  :", roc_auc_score(y_test_tfidf, y_score_tfidf))

In [ ]:
cm_tfidf = confusion_matrix(
    y_test_tfidf,
    y_pred_tfidf
)

print(cm_tfidf)

ConfusionMatrixDisplay(
    confusion_matrix=cm_tfidf,
    display_labels=["good", "bad"]
).plot()

In [ ]:
results = pd.DataFrame([
    {
        "representation": "LLVM stats",
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_score),
    },
    {
        "representation": "Opcode TF-IDF",
        "accuracy": accuracy_score(y_test_tfidf, y_pred_tfidf),
        "precision": precision_score(y_test_tfidf, y_pred_tfidf),
        "recall": recall_score(y_test_tfidf, y_pred_tfidf),
        "f1": f1_score(y_test_tfidf, y_pred_tfidf),
        "roc_auc": roc_auc_score(y_test_tfidf, y_score_tfidf),
    }
])

results

## структурные признаки CFG.

In [ ]:


def extract_cfg_features(path):
    text = Path(path).read_text(errors="ignore")
    lines = text.splitlines()

    functions = 0
    basic_blocks = 0
    edges = 0
    total_instructions = 0

    conditional_branches = 0
    unconditional_branches = 0
    switches = 0
    returns = 0

    inside_function = False

    for raw_line in lines:
        line = raw_line.strip()

        if line.startswith("define "):
            functions += 1
            inside_function = True
            continue

        if inside_function and line == "}":
            inside_function = False
            continue

        if not inside_function:
            continue

        if not line or line.startswith(";"):
            continue

        # Метка basic block:
        # entry:
        # 13:
        # if.then:
        if re.match(r"^[A-Za-z0-9$._-]+:\s*(;.*)?$", line):
            basic_blocks += 1
            continue

        # Убираем левую часть:
        # %12 = icmp ...
        if "=" in line:
            instruction = line.split("=", 1)[1].strip()
        else:
            instruction = line

        tokens = instruction.split()

        if not tokens:
            continue

        opcode = tokens[0]

        if opcode in OPCODE_SET:
            total_instructions += 1

        # conditional:
        # br i1 %cond, label %A, label %B
        if line.startswith("br i1 "):
            conditional_branches += 1
            edges += 2

        # unconditional:
        # br label %A
        elif line.startswith("br label "):
            unconditional_branches += 1
            edges += 1

        elif line.startswith("switch "):
            switches += 1

            # Каждый "label %" соответствует выходному ребру
            n_targets = len(re.findall(r"label\s+%", line))
            edges += n_targets

        elif line.startswith("ret "):
            returns += 1

    # В LLVM первый block функции иногда не имеет явной метки.
    # Поэтому как грубую структурную оценку добавляем по одному entry block
    # на функцию.
    nodes = basic_blocks + functions

    if nodes > 0:
        edges_per_node = edges / nodes
        instructions_per_block = total_instructions / nodes
    else:
        edges_per_node = 0.0
        instructions_per_block = 0.0

    if functions > 0:
        blocks_per_function = nodes / functions
    else:
        blocks_per_function = 0.0

    # Приближение цикломатической сложности:
    # M = E - N + 2P
    cyclomatic = edges - nodes + 2 * functions

    return {
        "functions": functions,
        "cfg_nodes": nodes,
        "cfg_edges": edges,
        "cyclomatic": cyclomatic,
        "instructions": total_instructions,
        "conditional_branches": conditional_branches,
        "unconditional_branches": unconditional_branches,
        "switches": switches,
        "returns": returns,
        "edges_per_node": edges_per_node,
        "instructions_per_block": instructions_per_block,
        "blocks_per_function": blocks_per_function,
    }

In [ ]:
cfg_example = extract_cfg_features(df.iloc[0]["path"])

cfg_example

In [ ]:
cfg_rows = []

for path in df["path"]:
    cfg_rows.append(
        extract_cfg_features(path)
    )

cfg_df = pd.DataFrame(cfg_rows)

cfg_df.head()

In [ ]:
X_cfg = cfg_df
y_cfg = df["target"]

In [ ]:
X_train_cfg = X_cfg.iloc[train_idx]
X_test_cfg = X_cfg.iloc[test_idx]

y_train_cfg = y_cfg.iloc[train_idx]
y_test_cfg = y_cfg.iloc[test_idx]

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train_cfg)

fig, ax = plt.subplots(figsize=(8, 6))

good = y_train == 0
bad = y_train == 1

ax.scatter(
    X_pca[good, 0],
    X_pca[good, 1],
    s=10,
    alpha=0.7,
    label="good"
)

ax.scatter(
    X_pca[bad, 0],
    X_pca[bad, 1],
    s=10,
    alpha=0.7,
    label="bad"
)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("PCA для CFG")
ax.legend()

plt.show()

In [ ]:
cfg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        C=1.0,
        max_iter=5000,
        random_state=42,
    )),
])

cfg_model.fit(
    X_train_cfg,
    y_train_cfg
)

In [ ]:
y_pred_cfg = cfg_model.predict(X_test_cfg)

y_score_cfg = cfg_model.predict_proba(
    X_test_cfg
)[:, 1]

In [ ]:
print("Accuracy :", accuracy_score(y_test_cfg, y_pred_cfg))
print("Precision:", precision_score(y_test_cfg, y_pred_cfg))
print("Recall   :", recall_score(y_test_cfg, y_pred_cfg))
print("F1       :", f1_score(y_test_cfg, y_pred_cfg))
print("ROC-AUC  :", roc_auc_score(y_test_cfg, y_score_cfg))

In [ ]:
cm_cfg = confusion_matrix(
    y_test_cfg,
    y_pred_cfg
)

ConfusionMatrixDisplay(
    confusion_matrix=cm_cfg,
    display_labels=["good", "bad"]
).plot()

In [ ]:
results = pd.DataFrame([
    {
        "representation": "LLVM stats",
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_score),
    },
    {
        "representation": "Opcode TF-IDF",
        "accuracy": accuracy_score(y_test_tfidf, y_pred_tfidf),
        "precision": precision_score(y_test_tfidf, y_pred_tfidf),
        "recall": recall_score(y_test_tfidf, y_pred_tfidf),
        "f1": f1_score(y_test_tfidf, y_pred_tfidf),
        "roc_auc": roc_auc_score(y_test_tfidf, y_score_tfidf),
    },
    {
        "representation": "CFG structural",
        "accuracy": accuracy_score(y_test_cfg, y_pred_cfg),
        "precision": precision_score(y_test_cfg, y_pred_cfg),
        "recall": recall_score(y_test_cfg, y_pred_cfg),
        "f1": f1_score(y_test_cfg, y_pred_cfg),
        "roc_auc": roc_auc_score(y_test_cfg, y_score_cfg),
    },
])

results

### Leave-One-CWE-Out

In [ ]:
loo_stats_results = []

for held_out_cwe in ["cwe23", "cwe78", "cwe134"]:

    train_mask = df["cwe"] != held_out_cwe
    test_mask = df["cwe"] == held_out_cwe

    X_train_loo = features_df.loc[train_mask]
    X_test_loo = features_df.loc[test_mask]

    y_train_loo = df.loc[train_mask, "target"]
    y_test_loo = df.loc[test_mask, "target"]

    model_loo = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            C=1.0,
            max_iter=5000,
            random_state=42,
        )),
    ])

    model_loo.fit(X_train_loo, y_train_loo)

    y_pred_loo = model_loo.predict(X_test_loo)
    y_score_loo = model_loo.predict_proba(X_test_loo)[:, 1]

    loo_stats_results.append({
        "representation": "LLVM stats",
        "held_out_cwe": held_out_cwe,
        "accuracy": accuracy_score(y_test_loo, y_pred_loo),
        "precision": precision_score(y_test_loo, y_pred_loo),
        "recall": recall_score(y_test_loo, y_pred_loo),
        "f1": f1_score(y_test_loo, y_pred_loo),
        "roc_auc": roc_auc_score(y_test_loo, y_score_loo),
    })

loo_stats_df = pd.DataFrame(loo_stats_results)

loo_stats_df

In [ ]:
loo_cfg_results = []

for held_out_cwe in ["cwe23", "cwe78", "cwe134"]:

    train_mask = df["cwe"] != held_out_cwe
    test_mask = df["cwe"] == held_out_cwe

    X_train_loo = cfg_df.loc[train_mask]
    X_test_loo = cfg_df.loc[test_mask]

    y_train_loo = df.loc[train_mask, "target"]
    y_test_loo = df.loc[test_mask, "target"]

    model_loo = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=1.0,
            max_iter=5000,
            random_state=42,
        )),
    ])

    model_loo.fit(X_train_loo, y_train_loo)

    y_pred_loo = model_loo.predict(X_test_loo)
    y_score_loo = model_loo.predict_proba(X_test_loo)[:, 1]

    loo_cfg_results.append({
        "representation": "CFG structural",
        "held_out_cwe": held_out_cwe,
        "accuracy": accuracy_score(y_test_loo, y_pred_loo),
        "precision": precision_score(y_test_loo, y_pred_loo),
        "recall": recall_score(y_test_loo, y_pred_loo),
        "f1": f1_score(y_test_loo, y_pred_loo),
        "roc_auc": roc_auc_score(y_test_loo, y_score_loo),
    })

loo_cfg_df = pd.DataFrame(loo_cfg_results)

loo_cfg_df

In [ ]:
loo_tfidf_results = []

for held_out_cwe in ["cwe23", "cwe78", "cwe134"]:

    train_mask = df["cwe"] != held_out_cwe
    test_mask = df["cwe"] == held_out_cwe

    train_indices = df.index[train_mask]
    test_indices = df.index[test_mask]

    train_texts = [opcode_texts[i] for i in train_indices]
    test_texts = [opcode_texts[i] for i in test_indices]

    y_train_loo = df.loc[train_mask, "target"]
    y_test_loo = df.loc[test_mask, "target"]

    tfidf_loo = TfidfVectorizer(
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        ngram_range=(1, 2),
        min_df=2,
    )

    X_train_loo = tfidf_loo.fit_transform(train_texts)
    X_test_loo = tfidf_loo.transform(test_texts)

    model_loo = LogisticRegression(
        C=1.0,
        max_iter=5000,
        random_state=42,
    )

    model_loo.fit(X_train_loo, y_train_loo)

    y_pred_loo = model_loo.predict(X_test_loo)
    y_score_loo = model_loo.predict_proba(X_test_loo)[:, 1]

    loo_tfidf_results.append({
        "representation": "Opcode TF-IDF",
        "held_out_cwe": held_out_cwe,
        "accuracy": accuracy_score(y_test_loo, y_pred_loo),
        "precision": precision_score(y_test_loo, y_pred_loo),
        "recall": recall_score(y_test_loo, y_pred_loo),
        "f1": f1_score(y_test_loo, y_pred_loo),
        "roc_auc": roc_auc_score(y_test_loo, y_score_loo),
    })

loo_tfidf_df = pd.DataFrame(loo_tfidf_results)

loo_tfidf_df

In [ ]:
loo_results = pd.concat([
    loo_stats_df,
    loo_tfidf_df,
    loo_cfg_df,
], ignore_index=True)

loo_results

In [ ]:
recall_table = loo_results.pivot(
    index="representation",
    columns="held_out_cwe",
    values="recall",
)

recall_table

## средний и худший Recall:

In [ ]:
loo_summary = (
    loo_results
    .groupby("representation")
    .agg(
        mean_recall=("recall", "mean"),
        min_recall=("recall", "min"),
        mean_f1=("f1", "mean"),
        min_f1=("f1", "min"),
        mean_auc=("roc_auc", "mean"),
    )
    .sort_values("mean_recall", ascending=False)
)

loo_summary

То есть при стандартном пороге 0.5 модель почти всех объявляет bad. Поэтому Recall идеальный, но классификатор при таком пороге плоховат.

Но для нашей задачи это не обязательно проблема, потому что мы хотим не столько сказать:
bad / good

сколько получить:
насколько подозрителен объект?

То есть использовать y_score и сортировать объекты.

И здесь ROC-AUC у TF-IDF хороший, особенно в среднем:

Opcode TF-IDF:
mean Recall ≈ 0.999
worst Recall ≈ 0.996

mean AUC ≈ 0.968

Поэтому именно Opcode TF-IDF + Logistic Regression логично сейчас проверить как ранжировщик.

Делаем главный эксперимент: Recall vs доля выбранных объектов.

In [ ]:
import math

fractions = [
    0.1, 0.2, 0.3, 0.4, 0.5,
    0.6, 0.7, 0.8, 0.9, 1.0
]

ranking_rows = []
ranking_auc_rows = []

for held_out_cwe in ["cwe23", "cwe78", "cwe134"]:

    train_mask = df["cwe"] != held_out_cwe
    test_mask = df["cwe"] == held_out_cwe

    train_indices = df.index[train_mask]
    test_indices = df.index[test_mask]

    train_texts = [opcode_texts[i] for i in train_indices]
    test_texts = [opcode_texts[i] for i in test_indices]

    y_train_loo = df.loc[train_mask, "target"].to_numpy()
    y_test_loo = df.loc[test_mask, "target"].to_numpy()

    # TF-IDF обучаем только на train
    tfidf_loo = TfidfVectorizer(
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        ngram_range=(1, 2),
        min_df=2,
    )

    X_train_loo = tfidf_loo.fit_transform(train_texts)
    X_test_loo = tfidf_loo.transform(test_texts)

    model_loo = LogisticRegression(
        C=1.0,
        max_iter=5000,
        random_state=42,
    )

    model_loo.fit(
        X_train_loo,
        y_train_loo
    )

    # Score: чем выше, тем сильнее модель считает объект bad
    scores = model_loo.predict_proba(
        X_test_loo
    )[:, 1]

    # ROC-AUC для полного held-out CWE
    auc = roc_auc_score(
        y_test_loo,
        scores
    )

    print(
        held_out_cwe,
        "ROC-AUC =",
        auc
    )

    ranking_auc_rows.append({
        "held_out_cwe": held_out_cwe,
        "roc_auc": auc,
    })

    # Сортируем от наиболее подозрительных к наименее
    order = np.argsort(-scores)

    y_sorted = y_test_loo[order]

    total_bad = y_test_loo.sum()

    for fraction in fractions:

        k = math.ceil(
            fraction * len(y_sorted)
        )

        selected = y_sorted[:k]

        found_bad = selected.sum()

        recall = found_bad / total_bad

        ranking_rows.append({
            "held_out_cwe": held_out_cwe,
            "fraction": fraction,
            "selected_objects": k,
            "found_bad": int(found_bad),
            "total_bad": int(total_bad),
            "recall": recall,
        })


# После завершения всех трёх LOO-экспериментов
ranking_df = pd.DataFrame(ranking_rows)

ranking_auc_df = pd.DataFrame(
    ranking_auc_rows
)

print()
print("ROC-AUC по CWE:")
display(ranking_auc_df)

print(
    "Mean ROC-AUC:",
    ranking_auc_df["roc_auc"].mean()
)

print(
    "Worst ROC-AUC:",
    ranking_auc_df["roc_auc"].min()
)

ranking_df

In [ ]:
recall_fraction_table = ranking_df.pivot(
    index="fraction",
    columns="held_out_cwe",
    values="recall",
)

recall_fraction_table

средний и худший Recall на каждой доле

In [ ]:
ranking_summary = (
    ranking_df
    .groupby("fraction")
    .agg(
        mean_recall=("recall", "mean"),
        worst_recall=("recall", "min"),
    )
    .reset_index()
)

ranking_summary

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

for cwe in ["cwe23", "cwe78", "cwe134"]:
    part = ranking_df[
        ranking_df["held_out_cwe"] == cwe
    ]

    plt.plot(
        part["fraction"],
        part["recall"],
        marker="o",
        label=cwe,
    )

plt.xlabel("Доля отобранных тестовых объектов")
plt.ylabel("Recall")
plt.title("Recall vs доля отобранных объектов")
plt.xticks(fractions)
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    ranking_summary["fraction"],
    ranking_summary["worst_recall"],
    marker="o",
)

plt.xlabel("Доля отобранных тестовых объектов")
plt.ylabel("Worst-case Recall")
plt.title("Худший Recall по Leave-One-CWE-Out")
plt.xticks(fractions)
plt.ylim(0, 1.05)
plt.grid(True)

plt.show()

In [ ]:
for threshold in [0.90, 0.95, 0.99]:

    candidates = ranking_summary[
        ranking_summary["worst_recall"] >= threshold
    ]

    if len(candidates) == 0:
        print(
            f"Recall >= {threshold}: "
            "не достигнут"
        )
    else:
        first = candidates.iloc[0]

        print(
            f"Recall >= {threshold}: "
            f"{int(first['fraction'] * 100)}% объектов, "
            f"worst recall = {first['worst_recall']:.4f}"
        )

In [ ]:
RESULTS_DIR = BASE / "results"
RESULTS_DIR.mkdir(exist_ok=True)

results.to_excel(
    RESULTS_DIR / "standard_split.xlsx",
    index=False,
)

loo_results.to_excel(
    RESULTS_DIR / "leave_one_cwe_out.xlsx",
    index=False,
)

ranking_df.to_excel(
    RESULTS_DIR / "ranking.xlsx",
    index=False,
)

ranking_summary.to_excel(
    RESULTS_DIR / "ranking_summary.xlsx",
    index=False,
)

ranking_auc_df.to_excel(
    RESULTS_DIR / "ranking_auc.xlsx",
    index=False,
)

print("Сохранено в:", RESULTS_DIR)

In [ ]:
def make_models():

    return {
        "LogReg": Pipeline([
            ("scaler", StandardScaler(with_mean=False)),
            ("model", LogisticRegression(
                C=1.0,
                max_iter=5000,
                random_state=42,
            )),
        ]),

        "SVM": Pipeline([
            ("scaler", StandardScaler(with_mean=False)),
            ("model", SVC(
                kernel="rbf",
                C=1.0,
                probability=True,
                random_state=42,
            )),
        ]),

        "XGBoost": XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            tree_method="hist",
            n_jobs=-1,
            random_state=42,
            eval_metric="logloss",
        ),

        "MLP": Pipeline([
            ("scaler", StandardScaler(with_mean=False)),
            ("model", MLPClassifier(
                hidden_layer_sizes=(64, 32),
                max_iter=1000,
                early_stopping=True,
                random_state=42,
            )),
        ]),
    }

In [ ]:
def evaluate_models(
    representation,
    X_train,
    X_test,
    y_train,
    y_test,
):

    rows = []

    models = make_models()

    for model_name, model in models.items():

        print(
            f"{representation} + {model_name}"
        )

        # Время обучения
        start = time.perf_counter()

        model.fit(
            X_train,
            y_train
        )

        train_time = time.perf_counter() - start

        # Время inference
        start = time.perf_counter()

        y_pred = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(
                X_test
            )[:, 1]
        else:
            y_score = model.decision_function(
                X_test
            )

        inference_time = (
            time.perf_counter() - start
        )

        inference_us_per_sample = (
            inference_time
            / len(y_test)
            * 1_000_000
        )

        rows.append({
            "representation": representation,
            "model": model_name,

            "accuracy":
                accuracy_score(y_test, y_pred),

            "precision":
                precision_score(y_test, y_pred),

            "recall":
                recall_score(y_test, y_pred),

            "f1":
                f1_score(y_test, y_pred),

            "roc_auc":
                roc_auc_score(y_test, y_score),

            "train_time_s":
                train_time,

            "inference_us_per_sample":
                inference_us_per_sample,
        })

    return rows

In [ ]:
X_stats = features_df

X_train_stats = X_stats.iloc[train_idx]
X_test_stats = X_stats.iloc[test_idx]

y_train_common = df["target"].iloc[train_idx]
y_test_common = df["target"].iloc[test_idx]

In [ ]:
X_cfg = cfg_df

X_train_cfg = X_cfg.iloc[train_idx]
X_test_cfg = X_cfg.iloc[test_idx]

In [ ]:
train_texts = [
    opcode_texts[i]
    for i in train_idx
]

test_texts = [
    opcode_texts[i]
    for i in test_idx
]

tfidf_standard = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=2,
)

X_train_tfidf = (
    tfidf_standard.fit_transform(
        train_texts
    )
)

X_test_tfidf = (
    tfidf_standard.transform(
        test_texts
    )
)

print(
    X_train_tfidf.shape,
    X_test_tfidf.shape
)

In [ ]:
all_results = []

all_results += evaluate_models(
    "LLVM stats",
    X_train_stats,
    X_test_stats,
    y_train_common,
    y_test_common,
)

all_results += evaluate_models(
    "Opcode TF-IDF",
    X_train_tfidf,
    X_test_tfidf,
    y_train_common,
    y_test_common,
)

all_results += evaluate_models(
    "CFG structural",
    X_train_cfg,
    X_test_cfg,
    y_train_common,
    y_test_common,
)

In [ ]:
benchmark_df = pd.DataFrame(
    all_results
)

benchmark_df

In [ ]:
benchmark_sorted = (
    benchmark_df
    .sort_values(
        ["recall", "f1", "roc_auc"],
        ascending=False,
    )
    .reset_index(drop=True)
)

benchmark_sorted

In [ ]:
standard_recall_table = (
    benchmark_df.pivot(
        index="representation",
        columns="model",
        values="recall",
    )
)

standard_recall_table

In [ ]:
standard_f1_table = (
    benchmark_df.pivot(
        index="representation",
        columns="model",
        values="f1",
    )
)

standard_f1_table

In [ ]:
standard_auc_table = (
    benchmark_df.pivot(
        index="representation",
        columns="model",
        values="roc_auc",
    )
)

standard_auc_table

## все 3 представления × 4 модели × 3 held-out CWE = 36 запусков

In [ ]:
def evaluate_numeric_loo(
    representation,
    X,
):

    rows = []

    for held_out_cwe in ["cwe23", "cwe78", "cwe134"]:

        print(
            f"\n{representation} | "
            f"TEST = {held_out_cwe}"
        )

        train_mask = df["cwe"] != held_out_cwe
        test_mask = df["cwe"] == held_out_cwe

        X_train = X.loc[train_mask]
        X_test = X.loc[test_mask]

        y_train = df.loc[
            train_mask,
            "target"
        ]

        y_test = df.loc[
            test_mask,
            "target"
        ]

        models = make_models()

        for model_name, model in models.items():

            print("  ", model_name)

            start = time.perf_counter()

            model.fit(
                X_train,
                y_train
            )

            train_time = (
                time.perf_counter()
                - start
            )

            start = time.perf_counter()

            y_pred = model.predict(
                X_test
            )

            if hasattr(model, "predict_proba"):

                y_score = model.predict_proba(
                    X_test
                )[:, 1]

            else:

                y_score = model.decision_function(
                    X_test
                )

            inference_time = (
                time.perf_counter()
                - start
            )

            rows.append({

                "representation":
                    representation,

                "model":
                    model_name,

                "held_out_cwe":
                    held_out_cwe,

                "accuracy":
                    accuracy_score(
                        y_test,
                        y_pred
                    ),

                "precision":
                    precision_score(
                        y_test,
                        y_pred,
                        zero_division=0
                    ),

                "recall":
                    recall_score(
                        y_test,
                        y_pred
                    ),

                "f1":
                    f1_score(
                        y_test,
                        y_pred
                    ),

                "roc_auc":
                    roc_auc_score(
                        y_test,
                        y_score
                    ),

                "train_time_s":
                    train_time,

                "inference_us_per_sample":
                    (
                        inference_time
                        / len(y_test)
                        * 1_000_000
                    ),
            })

    return rows

In [ ]:
def evaluate_tfidf_loo():

    rows = []

    for held_out_cwe in ["cwe23", "cwe78", "cwe134"]:

        print(
            f"\nOpcode TF-IDF | "
            f"TEST = {held_out_cwe}"
        )

        train_mask = df["cwe"] != held_out_cwe
        test_mask = df["cwe"] == held_out_cwe

        train_indices = df.index[
            train_mask
        ]

        test_indices = df.index[
            test_mask
        ]

        train_texts = [
            opcode_texts[i]
            for i in train_indices
        ]

        test_texts = [
            opcode_texts[i]
            for i in test_indices
        ]

        y_train = df.loc[
            train_mask,
            "target"
        ]

        y_test = df.loc[
            test_mask,
            "target"
        ]

        # Очень важно:
        # fit только на train
        tfidf = TfidfVectorizer(
            tokenizer=str.split,
            preprocessor=None,
            token_pattern=None,
            ngram_range=(1, 2),
            min_df=2,
        )

        X_train = tfidf.fit_transform(
            train_texts
        )

        X_test = tfidf.transform(
            test_texts
        )

        models = make_models()

        for model_name, model in models.items():

            print("  ", model_name)

            start = time.perf_counter()

            model.fit(
                X_train,
                y_train
            )

            train_time = (
                time.perf_counter()
                - start
            )

            start = time.perf_counter()

            y_pred = model.predict(
                X_test
            )

            if hasattr(model, "predict_proba"):

                y_score = model.predict_proba(
                    X_test
                )[:, 1]

            else:

                y_score = model.decision_function(
                    X_test
                )

            inference_time = (
                time.perf_counter()
                - start
            )

            rows.append({

                "representation":
                    "Opcode TF-IDF",

                "model":
                    model_name,

                "held_out_cwe":
                    held_out_cwe,

                "accuracy":
                    accuracy_score(
                        y_test,
                        y_pred
                    ),

                "precision":
                    precision_score(
                        y_test,
                        y_pred,
                        zero_division=0
                    ),

                "recall":
                    recall_score(
                        y_test,
                        y_pred
                    ),

                "f1":
                    f1_score(
                        y_test,
                        y_pred
                    ),

                "roc_auc":
                    roc_auc_score(
                        y_test,
                        y_score
                    ),

                "train_time_s":
                    train_time,

                "inference_us_per_sample":
                    (
                        inference_time
                        / len(y_test)
                        * 1_000_000
                    ),
            })

    return rows

In [ ]:
loo_all_rows = []

loo_all_rows += evaluate_numeric_loo(
    "LLVM stats",
    features_df,
)

loo_all_rows += evaluate_tfidf_loo()

loo_all_rows += evaluate_numeric_loo(
    "CFG structural",
    cfg_df,
)

In [ ]:
loo_all_df = pd.DataFrame(
    loo_all_rows
)

loo_all_df

In [ ]:
loo_recall_matrix = loo_all_df.pivot_table(
    index=[
        "representation",
        "model"
    ],
    columns="held_out_cwe",
    values="recall",
)

loo_recall_matrix

In [ ]:
loo_model_summary = (
    loo_all_df
    .groupby([
        "representation",
        "model"
    ])
    .agg(
        mean_recall=(
            "recall",
            "mean"
        ),
        min_recall=(
            "recall",
            "min"
        ),
        mean_f1=(
            "f1",
            "mean"
        ),
        min_f1=(
            "f1",
            "min"
        ),
        mean_auc=(
            "roc_auc",
            "mean"
        ),
        min_auc=(
            "roc_auc",
            "min"
        ),
        mean_train_time_s=(
            "train_time_s",
            "mean"
        ),
        mean_inference_us=(
            "inference_us_per_sample",
            "mean"
        ),
    )
    .reset_index()
    .sort_values(
        [
            "min_recall",
            "mean_recall"
        ],
        ascending=False,
    )
)

loo_model_summary

In [ ]:
loo_auc_matrix = loo_all_df.pivot_table(
    index=[
        "representation",
        "model"
    ],
    columns="held_out_cwe",
    values="roc_auc",
)

loo_auc_matrix

In [ ]:
loo_f1_matrix = loo_all_df.pivot_table(
    index=[
        "representation",
        "model"
    ],
    columns="held_out_cwe",
    values="f1",
)

loo_f1_matrix

In [ ]:
loo_all_df.to_excel(
    RESULTS_DIR / "loo_12_models.xlsx",
    index=False,
)

loo_model_summary.to_excel(
    RESULTS_DIR / "loo_12_models_summary.xlsx",
    index=False,
)

In [ ]:
def get_ranking_scores(model, X):
    if hasattr(model, "decision_function"):
        return model.decision_function(X)

    return model.predict_proba(X)[:, 1]

In [ ]:
def make_ranking_models():

    return {
        "LogReg": Pipeline([
            ("scaler", StandardScaler(with_mean=False)),
            ("model", LogisticRegression(
                C=1.0,
                max_iter=5000,
                random_state=42,
            )),
        ]),

        "SVM": Pipeline([
            ("scaler", StandardScaler(with_mean=False)),
            ("model", SVC(
                kernel="rbf",
                C=1.0,
                probability=False,
                random_state=42,
            )),
        ]),

        "XGBoost": XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            tree_method="hist",
            n_jobs=-1,
            random_state=42,
            eval_metric="logloss",
        ),

        "MLP": Pipeline([
            ("scaler", StandardScaler(with_mean=False)),
            ("model", MLPClassifier(
                hidden_layer_sizes=(64, 32),
                max_iter=1000,
                early_stopping=True,
                random_state=42,
            )),
        ]),
    }

In [ ]:
fractions = [
    0.1, 0.2, 0.3, 0.4, 0.5,
    0.6, 0.7, 0.8, 0.9, 1.0
]


def ranking_numeric_loo(
    representation,
    X,
):

    rows = []

    for held_out_cwe in [
        "cwe23",
        "cwe78",
        "cwe134"
    ]:

        train_mask = df["cwe"] != held_out_cwe
        test_mask = df["cwe"] == held_out_cwe

        X_train = X.loc[train_mask]
        X_test = X.loc[test_mask]

        y_train = df.loc[
            train_mask,
            "target"
        ].to_numpy()

        y_test = df.loc[
            test_mask,
            "target"
        ].to_numpy()

        models = make_ranking_models()

        for model_name, model in models.items():

            print(
                representation,
                model_name,
                held_out_cwe
            )

            model.fit(
                X_train,
                y_train
            )

            scores = get_ranking_scores(
                model,
                X_test
            )

            auc = roc_auc_score(
                y_test,
                scores
            )

            order = np.argsort(-scores)

            y_sorted = y_test[order]

            total_bad = y_test.sum()

            for fraction in fractions:

                k = math.ceil(
                    fraction * len(y_sorted)
                )

                found_bad = (
                    y_sorted[:k].sum()
                )

                rows.append({
                    "representation":
                        representation,

                    "model":
                        model_name,

                    "held_out_cwe":
                        held_out_cwe,

                    "fraction":
                        fraction,

                    "recall":
                        found_bad / total_bad,

                    "roc_auc":
                        auc,
                })

    return rows

In [ ]:
def ranking_tfidf_loo():

    rows = []

    for held_out_cwe in [
        "cwe23",
        "cwe78",
        "cwe134"
    ]:

        train_mask = df["cwe"] != held_out_cwe
        test_mask = df["cwe"] == held_out_cwe

        train_indices = df.index[
            train_mask
        ]

        test_indices = df.index[
            test_mask
        ]

        train_texts = [
            opcode_texts[i]
            for i in train_indices
        ]

        test_texts = [
            opcode_texts[i]
            for i in test_indices
        ]

        y_train = df.loc[
            train_mask,
            "target"
        ].to_numpy()

        y_test = df.loc[
            test_mask,
            "target"
        ].to_numpy()

        tfidf = TfidfVectorizer(
            tokenizer=str.split,
            preprocessor=None,
            token_pattern=None,
            ngram_range=(1, 2),
            min_df=2,
        )

        X_train = tfidf.fit_transform(
            train_texts
        )

        X_test = tfidf.transform(
            test_texts
        )

        models = make_ranking_models()

        for model_name, model in models.items():

            print(
                "Opcode TF-IDF",
                model_name,
                held_out_cwe
            )

            model.fit(
                X_train,
                y_train
            )

            scores = get_ranking_scores(
                model,
                X_test
            )

            auc = roc_auc_score(
                y_test,
                scores
            )

            order = np.argsort(-scores)

            y_sorted = y_test[order]

            total_bad = y_test.sum()

            for fraction in fractions:

                k = math.ceil(
                    fraction * len(y_sorted)
                )

                found_bad = (
                    y_sorted[:k].sum()
                )

                rows.append({
                    "representation":
                        "Opcode TF-IDF",

                    "model":
                        model_name,

                    "held_out_cwe":
                        held_out_cwe,

                    "fraction":
                        fraction,

                    "recall":
                        found_bad / total_bad,

                    "roc_auc":
                        auc,
                })

    return rows

In [ ]:
ranking_all_rows = []

ranking_all_rows += ranking_numeric_loo(
    "LLVM stats",
    features_df,
)

ranking_all_rows += ranking_tfidf_loo()

ranking_all_rows += ranking_numeric_loo(
    "CFG structural",
    cfg_df,
)

ranking_all_df = pd.DataFrame(
    ranking_all_rows
)

ranking_all_df.shape

In [ ]:
ranking_all_summary = (
    ranking_all_df
    .groupby([
        "representation",
        "model",
        "fraction",
    ])
    .agg(
        mean_recall=(
            "recall",
            "mean"
        ),
        worst_recall=(
            "recall",
            "min"
        ),
        mean_auc=(
            "roc_auc",
            "mean"
        ),
        worst_auc=(
            "roc_auc",
            "min"
        ),
    )
    .reset_index()
)

ranking_all_summary

In [ ]:
ranking_worst_table = (
    ranking_all_summary
    .pivot_table(
        index=[
            "representation",
            "model"
        ],
        columns="fraction",
        values="worst_recall",
    )
)

ranking_worst_table

In [ ]:
compare_60_80 = (
    ranking_all_summary[
        ranking_all_summary["fraction"].isin([0.6, 0.7, 0.8])
    ]
    .pivot_table(
        index=["representation", "model"],
        columns="fraction",
        values="worst_recall",
    )
    .rename(columns={
        0.6: "worst_recall_60",
        0.7: "worst_recall_70",
        0.8: "worst_recall_80",
    })
    .sort_values(
        ["worst_recall_60", "worst_recall_70"],
        ascending=False,
    )
)

compare_60_80

In [ ]:
threshold_rows = []

for (representation, model), group in ranking_all_summary.groupby(
    ["representation", "model"]
):

    group = group.sort_values("fraction")

    row = {
        "representation": representation,
        "model": model,
    }

    for threshold in [0.90, 0.95, 0.99]:

        candidates = group[
            group["worst_recall"] >= threshold
        ]

        key = f"fraction_for_{int(threshold * 100)}_recall"

        if len(candidates) == 0:
            row[key] = np.nan
        else:
            row[key] = candidates.iloc[0]["fraction"]

    threshold_rows.append(row)

threshold_df = pd.DataFrame(threshold_rows)

threshold_df.sort_values(
    [
        "fraction_for_90_recall",
        "fraction_for_95_recall",
        "fraction_for_99_recall",
    ]
)

In [ ]:
model_choice = (
    compare_60_80
    .reset_index()
    .merge(
        loo_model_summary[
            [
                "representation",
                "model",
                "mean_recall",
                "min_recall",
                "mean_f1",
                "min_f1",
                "mean_auc",
                "min_auc",
                "mean_inference_us",
            ]
        ],
        on=["representation", "model"],
        how="left",
    )
)

model_choice = model_choice.sort_values(
    [
        "worst_recall_60",
        "worst_recall_70",
        "mean_auc",
    ],
    ascending=False,
)

model_choice

In [ ]:
top4 = (
    compare_60_80
    .reset_index()
    .head(4)[
        ["representation", "model"]
    ]
)

top4

In [ ]:
plt.figure(figsize=(9, 6))

for _, config in top4.iterrows():

    rep = config["representation"]
    model = config["model"]

    part = ranking_all_summary[
        (ranking_all_summary["representation"] == rep)
        &
        (ranking_all_summary["model"] == model)
    ]

    plt.plot(
        part["fraction"] * 100,
        part["worst_recall"],
        marker="o",
        label=f"{rep} + {model}",
    )

plt.xlabel("Доля отобранных объектов, %")
plt.ylabel("Worst-case Recall")
plt.title("Recall vs доля отбора: лучшие конфигурации")
plt.xticks(range(10, 101, 10))
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
ranking_all_df.to_excel(
    RESULTS_DIR / "ranking_all_models.xlsx",
    index=False,
)

ranking_all_summary.to_excel(
    RESULTS_DIR / "ranking_all_models_summary.xlsx",
    index=False,
)

compare_60_80.to_excel(
    RESULTS_DIR / "ranking_60_70_80.xlsx"
)

threshold_df.to_excel(
    RESULTS_DIR / "ranking_thresholds.xlsx",
    index=False,
)

model_choice.to_excel(
    RESULTS_DIR / "model_choice.xlsx",
    index=False,
)

In [ ]:
model_names_ru = {
    "LogReg": "Логистическая регрессия",
    "SVM": "SVM",
    "XGBoost": "XGBoost",
    "MLP": "MLP",
}

representation_names_ru = {
    "LLVM stats": "Статистические признаки LLVM",
    "Opcode TF-IDF": "TF-IDF последовательностей opcode",
    "CFG structural": "Структурные признаки CFG",
}

cwe_names_ru = {
    "cwe23": "CWE-23",
    "cwe78": "CWE-78",
    "cwe134": "CWE-134",
}

for representation in [
    "LLVM stats",
    "Opcode TF-IDF",
    "CFG structural"
]:

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, held_out_cwe in zip(
        axes,
        ["cwe23", "cwe78", "cwe134"]
    ):

        train_mask = df["cwe"] != held_out_cwe
        test_mask = df["cwe"] == held_out_cwe

        y_train_roc = df.loc[
            train_mask,
            "target"
        ].to_numpy()

        y_test_roc = df.loc[
            test_mask,
            "target"
        ].to_numpy()

        if representation == "LLVM stats":

            X_train_roc = features_df.loc[train_mask]
            X_test_roc = features_df.loc[test_mask]

        elif representation == "CFG structural":

            X_train_roc = cfg_df.loc[train_mask]
            X_test_roc = cfg_df.loc[test_mask]

        else:

            train_indices = df.index[train_mask]
            test_indices = df.index[test_mask]

            train_texts = [
                opcode_texts[i]
                for i in train_indices
            ]

            test_texts = [
                opcode_texts[i]
                for i in test_indices
            ]

            tfidf_roc = TfidfVectorizer(
                tokenizer=str.split,
                preprocessor=None,
                token_pattern=None,
                ngram_range=(1, 2),
                min_df=2,
            )

            X_train_roc = tfidf_roc.fit_transform(
                train_texts
            )

            X_test_roc = tfidf_roc.transform(
                test_texts
            )

        models = make_ranking_models()

        for model_name, model in models.items():

            model.fit(
                X_train_roc,
                y_train_roc
            )

            scores = get_ranking_scores(
                model,
                X_test_roc
            )

            fpr, tpr, _ = roc_curve(
                y_test_roc,
                scores
            )

            auc = roc_auc_score(
                y_test_roc,
                scores
            )

            ax.plot(
                fpr,
                tpr,
                label=(
                    f"{model_names_ru[model_name]} "
                    f"(AUC = {auc:.3f})"
                )
            )

        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            label="Случайный классификатор"
        )

        ax.set_title(
            f"Тест: {cwe_names_ru[held_out_cwe]}"
        )

        ax.set_xlabel(
            "Доля ложноположительных результатов"
        )

        ax.set_ylabel(
            "Доля истинноположительных результатов"
        )

        ax.grid()
        ax.legend()

    fig.suptitle(
        "ROC-кривые — "
        + representation_names_ru[representation],
        fontsize=14
    )

    plt.tight_layout()
    plt.show()

In [ ]:
model_names_ru = {
    "LogReg": "Логистическая регрессия",
    "SVM": "SVM",
    "XGBoost": "XGBoost",
    "MLP": "MLP",
}

representation_names_ru = {
    "LLVM stats": "Статистические признаки LLVM",
    "Opcode TF-IDF": "TF-IDF последовательностей opcode",
    "CFG structural": "Структурные признаки CFG",
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, representation in zip(
    axes,
    ["LLVM stats", "Opcode TF-IDF", "CFG structural"]
):

    y_train_roc = df.iloc[train_idx]["target"].to_numpy()
    y_test_roc = df.iloc[test_idx]["target"].to_numpy()

    if representation == "LLVM stats":

        X_train_roc = features_df.iloc[train_idx]
        X_test_roc = features_df.iloc[test_idx]

    elif representation == "CFG structural":

        X_train_roc = cfg_df.iloc[train_idx]
        X_test_roc = cfg_df.iloc[test_idx]

    else:

        train_texts = [
            opcode_texts[i]
            for i in train_idx
        ]

        test_texts = [
            opcode_texts[i]
            for i in test_idx
        ]

        tfidf_roc = TfidfVectorizer(
            tokenizer=str.split,
            preprocessor=None,
            token_pattern=None,
            ngram_range=(1, 2),
            min_df=2,
        )

        X_train_roc = tfidf_roc.fit_transform(
            train_texts
        )

        X_test_roc = tfidf_roc.transform(
            test_texts
        )

    models = make_ranking_models()

    for model_name, model in models.items():

        model.fit(
            X_train_roc,
            y_train_roc
        )

        scores = get_ranking_scores(
            model,
            X_test_roc
        )

        fpr, tpr, _ = roc_curve(
            y_test_roc,
            scores
        )

        auc = roc_auc_score(
            y_test_roc,
            scores
        )

        ax.plot(
            fpr,
            tpr,
            label=(
                f"{model_names_ru[model_name]} "
                f"(AUC = {auc:.3f})"
            )
        )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Случайный классификатор"
    )

    ax.set_title(
        representation_names_ru[representation]
    )

    ax.set_xlabel(
        "Доля ложноположительных результатов"
    )

    ax.set_ylabel(
        "Доля истинноположительных результатов"
    )

    ax.grid()
    ax.legend()

fig.suptitle(
    "ROC-кривые для стандартного разбиения",
    fontsize=14
)

plt.tight_layout()
plt.show()

### Замеряю скорости

In [ ]:
test_paths = df.iloc[test_idx]["path"].tolist()

times = []

for _ in range(5):
    start = time.perf_counter()

    rows = [
        extract_llvm_stats(path)
        for path in test_paths
    ]

    elapsed = time.perf_counter() - start
    times.append(elapsed / len(test_paths) * 1000)
llvm_mean = np.mean(times[1:])
llvm_std = np.std(times[1:])
llvm_times = times[1:]
print("LLVM stats, мс/образец:", np.mean(times))
print("std:", np.std(times))

In [ ]:
times = []

for _ in range(5):
    start = time.perf_counter()

    rows = [
        extract_cfg_features(path)
        for path in test_paths
    ]

    elapsed = time.perf_counter() - start
    times.append(elapsed / len(test_paths) * 1000)
cfg_mean = np.mean(times[1:])
cfg_std = np.std(times[1:])
cfg_times = times[1:]
print("CFG structural, мс/образец:", np.mean(times))
print("std:", np.std(times))

In [ ]:
train_paths = df.iloc[train_idx]["path"].tolist()

train_opcode_texts = [
    " ".join(extract_opcode_sequence(path))
    for path in train_paths
]

tfidf_time = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=2,
)

tfidf_time.fit(train_opcode_texts)

In [ ]:
times = []

for _ in range(5):

    start = time.perf_counter()

    test_opcode_texts = [
        " ".join(extract_opcode_sequence(path))
        for path in test_paths
    ]

    X_test_time = tfidf_time.transform(
        test_opcode_texts
    )

    elapsed = time.perf_counter() - start

    times.append(
        elapsed / len(test_paths) * 1000
    )
tfidf_mean = np.mean(times[1:])
tfidf_std = np.std(times[1:])
tfidf_times = times[1:]
print("Opcode TF-IDF, мс/образец:", np.mean(times))
print("std:", np.std(times))

In [ ]:
names = [
    "LLVM statistics",
    "Opcode TF-IDF",
    "CFG structural",
]

means = [
    llvm_mean,
    tfidf_mean,
    cfg_mean,
]

stds = [
    llvm_std,
    tfidf_std,
    cfg_std,
]

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    names,
    means,
    yerr=stds,
    capsize=5,
)

ax.set_ylabel("Время, мс/образец")
ax.set_title("Стоимость извлечения признаков")
ax.grid(axis="y", alpha=0.3)

for bar, value in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.3f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, len(llvm_times) + 1),
    llvm_times,
    marker="o",
    label="Статистические признаки LLVM"
)

plt.plot(
    range(1, len(tfidf_times) + 1),
    tfidf_times,
    marker="o",
    label="Opcode TF-IDF"
)

plt.plot(
    range(1, len(cfg_times) + 1),
    cfg_times,
    marker="o",
    label="Структурные признаки CFG"
)

plt.xlabel("Номер прогона")
plt.ylabel("Время, мс/образец")
plt.title("Время извлечения признаков по прогонам")

plt.grid()
plt.legend()

plt.show()

In [ ]:

# копия таблицы, чтобы не портить исходную
plot_df = benchmark_df.copy()

# русские подписи представлений
repr_map = {
    "LLVM stats": "Статистические признаки LLVM",
    "Opcode TF-IDF": "Opcode TF-IDF",
    "CFG structural": "Структурные признаки CFG",
}

# если модель у тебя называется SVM_RBF, замени на SVM
model_map = {
    "LogReg": "LogReg",
    "SVM": "SVM",
    "SVM_RBF": "SVM",
    "XGBoost": "XGBoost",
    "MLP": "MLP",
}

plot_df["representation_ru"] = plot_df["representation"].map(repr_map).fillna(plot_df["representation"])
plot_df["model_ru"] = plot_df["model"].map(model_map).fillna(plot_df["model"])

# подпись конфигурации на оси X
plot_df["config"] = (
    plot_df["representation_ru"] + "\n" + plot_df["model_ru"]
)

# можно отсортировать по Recall
plot_df = plot_df.sort_values("recall", ascending=False).reset_index(drop=True)

x = np.arange(len(plot_df))
width = 0.38

fig, ax = plt.subplots(figsize=(16, 7))

bars1 = ax.bar(
    x - width/2,
    plot_df["recall"],
    width,
    label="Recall"
)

bars2 = ax.bar(
    x + width/2,
    plot_df["f1"],
    width,
    label="F1"
)

ax.set_title("Качество моделей для разных представлений LLVM IR")
ax.set_ylabel("Значение метрики")
ax.set_xlabel("Конфигурация")
ax.set_xticks(x)
ax.set_xticklabels(plot_df["config"], rotation=45, ha="right")
ax.set_ylim(0.93, 1.01)   # можешь убрать, если хочешь полную шкалу от 0
ax.grid(axis="y", alpha=0.3)
ax.legend()

# подписи значений над столбцами
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width()/2,
            h + 0.001,
            f"{h:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90
        )

plt.tight_layout()
plt.show()

In [ ]:

plot_df = loo_recall_matrix.copy().reset_index()



model_map = {
    "LogReg": "LogReg",
    "SVM": "SVM",
    "SVM_RBF": "SVM",
    "XGBoost": "XGBoost",
    "MLP": "MLP",
}

plot_df["repr_short"] = plot_df["representation"].map(repr_map).fillna(plot_df["representation"])
plot_df["model_short"] = plot_df["model"].map(model_map).fillna(plot_df["model"])

plot_df["config"] = plot_df["repr_short"] + " + " + plot_df["model_short"]

# порядок столбцов
cwe_cols = ["cwe134", "cwe23", "cwe78"]

# если хочешь русские подписи CWE в легенде
legend_labels = {
    "cwe134": "CWE-134",
    "cwe23": "CWE-23",
    "cwe78": "CWE-78",
}

x = np.arange(len(plot_df))
width = 0.24

fig, ax = plt.subplots(figsize=(16, 7))

bars1 = ax.bar(
    x - width,
    plot_df["cwe134"],
    width,
    label="CWE-134"
)

bars2 = ax.bar(
    x,
    plot_df["cwe23"],
    width,
    label="CWE-23"
)

bars3 = ax.bar(
    x + width,
    plot_df["cwe78"],
    width,
    label="CWE-78"
)

ax.set_title("Recall при тестировании на удерживаемом CWE")
ax.set_ylabel("Recall")
ax.set_xlabel("Конфигурация")
ax.set_xticks(x)
ax.set_xticklabels(plot_df["config"], rotation=45, ha="right")
ax.set_ylim(0.75, 1.02)
ax.grid(axis="y", alpha=0.3)
ax.legend()

# подписи значений над столбцами
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + 0.003,
            f"{h:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90
        )

plt.tight_layout()
plt.show()

In [ ]:
repr_map = {
    "LLVM stats": "Статистические признаки LLVM",
    "Opcode TF-IDF": "Opcode TF-IDF",
    "CFG structural": "Структурные признаки CFG",
}

model_map = {
    "LogReg": "LogReg",
    "SVM": "SVM",
    "XGBoost": "XGBoost",
    "MLP": "MLP",
}

top4 = (
    compare_60_80
    .reset_index()
    .head(4)[["representation", "model"]]
)

plt.figure(figsize=(9, 6))

for _, config in top4.iterrows():

    rep = config["representation"]
    model = config["model"]

    part = ranking_all_summary[
        (ranking_all_summary["representation"] == rep) &
        (ranking_all_summary["model"] == model)
    ].sort_values("fraction")

    plt.plot(
        part["fraction"] * 100,
        part["worst_recall"],
        marker="o",
        label=f"{repr_map[rep]} + {model_map[model]}"
    )

# случайное ранжирование:
# если выбираем x% объектов случайно,
# ожидаемый Recall примерно x%
plt.plot(
    [10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
    [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    linestyle="--",
    label="Случайное ранжирование"
)

plt.xlabel("Доля отобранных объектов, %")
plt.ylabel("Минимальный Recall")
plt.title("Качество ранжирования лучших конфигураций")
plt.xticks(range(10, 101, 10))
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()